In [ ]:
# SQLite schema implementation for the DWBI data warehouse

import sqlite3
from pathlib import Path

# Create/connect to SQLite database file
db_path = "/content/hospital_dw.db"
conn = sqlite3.connect(db_path)
cur = conn.cursor()

# Enable foreign keys
cur.execute("PRAGMA foreign_keys = ON;")

# Drop existing objects if they already exist
drop_sql = """
DROP VIEW IF EXISTS vw_q1_ward_year_quarter;
DROP VIEW IF EXISTS vw_q2_ward_year_quarter_secondarydiagnosis;

DROP TABLE IF EXISTS fact_admission;
DROP TABLE IF EXISTS dim_diagnosis;
DROP TABLE IF EXISTS dim_consultant;
DROP TABLE IF EXISTS dim_ward;
DROP TABLE IF EXISTS dim_time;
"""
cur.executescript(drop_sql)

# Create schema
schema_sql = """
PRAGMA foreign_keys = ON;

-- =========================
-- DIMENSIONS
-- =========================

CREATE TABLE dim_time (
    time_key        INTEGER PRIMARY KEY,
    full_date       TEXT NOT NULL UNIQUE,      -- YYYY-MM-DD
    month_num       INTEGER NOT NULL CHECK (month_num BETWEEN 1 AND 12),
    quarter_num     INTEGER NOT NULL CHECK (quarter_num BETWEEN 1 AND 4),
    year_num        INTEGER NOT NULL
);

CREATE TABLE dim_ward (
    ward_key        INTEGER PRIMARY KEY,
    ward_id_bk      INTEGER NOT NULL UNIQUE,
    ward_name       TEXT NOT NULL
);

CREATE TABLE dim_consultant (
    consultant_key      INTEGER PRIMARY KEY,
    consultant_name     TEXT NOT NULL UNIQUE
);

CREATE TABLE dim_diagnosis (
    diagnosis_key       INTEGER PRIMARY KEY,
    diagnosis_name      TEXT NOT NULL UNIQUE
);

-- =========================
-- FACT TABLE
-- Grain: one row per admission
-- =========================

CREATE TABLE fact_admission (
    admission_key                   INTEGER PRIMARY KEY,
    admission_id_bk                 INTEGER NOT NULL UNIQUE,

    time_key                        INTEGER NOT NULL,
    ward_key                        INTEGER NOT NULL,
    consultant_key                  INTEGER,
    diagnosis_key                   INTEGER,

    -- Measures
    number_of_admissions            INTEGER NOT NULL DEFAULT 1 CHECK (number_of_admissions >= 0),
    length_of_stay                  INTEGER NOT NULL CHECK (length_of_stay >= 0),
    admission_cost                  REAL NOT NULL CHECK (admission_cost >= 0),

    total_operation_charges         REAL NOT NULL DEFAULT 0 CHECK (total_operation_charges >= 0),
    number_of_operations            INTEGER NOT NULL DEFAULT 0 CHECK (number_of_operations >= 0),

    secondary_diagnosis_days        INTEGER NOT NULL DEFAULT 0 CHECK (secondary_diagnosis_days >= 0),
    secondary_diagnosis_cost        REAL NOT NULL DEFAULT 0 CHECK (secondary_diagnosis_cost >= 0),

    FOREIGN KEY (time_key) REFERENCES dim_time(time_key),
    FOREIGN KEY (ward_key) REFERENCES dim_ward(ward_key),
    FOREIGN KEY (consultant_key) REFERENCES dim_consultant(consultant_key),
    FOREIGN KEY (diagnosis_key) REFERENCES dim_diagnosis(diagnosis_key)
);

-- =========================
-- INDEXES
-- =========================

CREATE INDEX idx_fact_admission_time
    ON fact_admission (time_key);

CREATE INDEX idx_fact_admission_ward
    ON fact_admission (ward_key);

CREATE INDEX idx_fact_admission_time_ward
    ON fact_admission (time_key, ward_key);

CREATE INDEX idx_fact_admission_consultant
    ON fact_admission (consultant_key);

CREATE INDEX idx_fact_admission_diagnosis
    ON fact_admission (diagnosis_key);

-- =========================
-- VIEWS FOR Q1 AND Q2
-- =========================

CREATE VIEW vw_q1_ward_year_quarter AS
SELECT
    w.ward_name,
    t.year_num,
    t.quarter_num,
    CASE
        WHEN SUM(f.number_of_operations) = 0 THEN NULL
        ELSE SUM(f.total_operation_charges) * 1.0 / SUM(f.number_of_operations)
    END AS charges_per_operation,
    SUM(f.length_of_stay) AS length_of_stay,
    SUM(f.admission_cost) AS admissions_cost
FROM fact_admission f
JOIN dim_time t ON f.time_key = t.time_key
JOIN dim_ward w ON f.ward_key = w.ward_key
GROUP BY w.ward_name, t.year_num, t.quarter_num;

CREATE VIEW vw_q2_ward_year_quarter_secondarydiagnosis AS
SELECT
    w.ward_name,
    t.year_num,
    t.quarter_num,
    CASE
        WHEN SUM(f.number_of_operations) = 0 THEN NULL
        ELSE SUM(f.total_operation_charges) * 1.0 / SUM(f.number_of_operations)
    END AS charges_per_operation,
    SUM(f.length_of_stay) AS length_of_stay,
    SUM(f.admission_cost) AS admissions_cost,
    SUM(f.secondary_diagnosis_days) AS secondary_diagnosis_days,
    SUM(f.secondary_diagnosis_cost) AS secondary_diagnosis_cost
FROM fact_admission f
JOIN dim_time t ON f.time_key = t.time_key
JOIN dim_ward w ON f.ward_key = w.ward_key
WHERE f.secondary_diagnosis_days > 0
   OR f.secondary_diagnosis_cost > 0
GROUP BY w.ward_name, t.year_num, t.quarter_num;
"""
cur.executescript(schema_sql)
conn.commit()

# Show created tables and views
print("Schema created successfully at:", db_path)
print("\nTables and views:")
for row in cur.execute("""
    SELECT name, type
    FROM sqlite_master
    WHERE type IN ('table', 'view')
    ORDER BY type, name;
"""):
    print(row)

print("\nDone.")
conn.close()

In [ ]:
# Assumes the DW schema/tables already exist.
# This cell only:
# 1) inserts sample data
# 2) builds the materialized-view-style table
# 3) prints Q1 and Q2 results

import sqlite3
import pandas as pd
from IPython.display import display

# IMPORTANT:
# Use the SAME database path you used when creating the schema.
db_path = "/content/hospital_dw.db"   # change only if your schema used a different file
conn = sqlite3.connect(db_path)
cur = conn.cursor()

cur.execute("PRAGMA foreign_keys = ON;")

# ------------------------------------------------------------
# 1) CLEAR OLD SAMPLE DATA / OLD MATERIALIZED VIEW TABLE
# ------------------------------------------------------------
cur.executescript("""
DROP TABLE IF EXISTS mv_ward_year_quarter;

DELETE FROM fact_admission;
DELETE FROM dim_diagnosis;
DELETE FROM dim_consultant;
DELETE FROM dim_ward;
DELETE FROM dim_time;
""")

# ------------------------------------------------------------
# 2) INSERT SAMPLE DATA
# ------------------------------------------------------------
cur.executemany("""
INSERT INTO dim_time (time_key, full_date, month_num, quarter_num, year_num)
VALUES (?, ?, ?, ?, ?)
""", [
    (1, "2024-01-15", 1, 1, 2024),
    (2, "2024-02-20", 2, 1, 2024),
    (3, "2024-04-10", 4, 2, 2024),
    (4, "2025-01-12", 1, 1, 2025),
])

cur.executemany("""
INSERT INTO dim_ward (ward_key, ward_id_bk, ward_name)
VALUES (?, ?, ?)
""", [
    (1, 101, "Cardiology"),
    (2, 102, "Orthopaedics"),
])

cur.executemany("""
INSERT INTO dim_consultant (consultant_key, consultant_name)
VALUES (?, ?)
""", [
    (1, "Dr Adams"),
    (2, "Dr Brown"),
])

cur.executemany("""
INSERT INTO dim_diagnosis (diagnosis_key, diagnosis_name)
VALUES (?, ?)
""", [
    (1, "Hypertension"),
    (2, "Fracture"),
    (3, "Diabetes"),
])

cur.executemany("""
INSERT INTO fact_admission (
    admission_key,
    admission_id_bk,
    time_key,
    ward_key,
    consultant_key,
    diagnosis_key,
    number_of_admissions,
    length_of_stay,
    admission_cost,
    total_operation_charges,
    number_of_operations,
    secondary_diagnosis_days,
    secondary_diagnosis_cost
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
""", [
    # 2024 Q1 - Cardiology
    (1, 1001, 1, 1, 1, 1, 1, 5, 1200.00, 500.00, 1, 0,   0.00),
    (2, 1002, 2, 1, 1, 3, 1, 7, 1800.00, 900.00, 2, 3, 400.00),

    # 2024 Q1 - Orthopaedics
    (3, 1003, 2, 2, 2, 2, 1, 4, 1500.00, 700.00, 1, 2, 250.00),

    # 2024 Q2 - Cardiology
    (4, 1004, 3, 1, 2, 1, 1, 6, 2100.00, 800.00, 2, 0,   0.00),

    # 2024 Q2 - Orthopaedics
    (5, 1005, 3, 2, 2, 2, 1, 9, 3200.00, 1500.00, 3, 4, 600.00),

    # 2025 Q1 - Cardiology
    (6, 1006, 4, 1, 1, 3, 1, 8, 2500.00, 1000.00, 2, 5, 900.00),
])

conn.commit()

# ------------------------------------------------------------
# 3) BUILD MATERIALIZED-VIEW-STYLE TABLE
# SQLite has no native materialized views, so use a precomputed table.
# ------------------------------------------------------------
cur.executescript("""
CREATE TABLE mv_ward_year_quarter AS
SELECT
    w.ward_key,
    w.ward_name,
    t.year_num,
    t.quarter_num,

    -- Q1 aggregates
    SUM(f.total_operation_charges) AS q1_sum_operation_charges,
    SUM(f.number_of_operations)    AS q1_sum_number_of_operations,
    SUM(f.length_of_stay)          AS q1_sum_length_of_stay,
    SUM(f.admission_cost)          AS q1_sum_admission_cost,

    -- Q2 aggregates (only admissions with secondary diagnosis activity)
    SUM(CASE
            WHEN f.secondary_diagnosis_days > 0 OR f.secondary_diagnosis_cost > 0
            THEN f.total_operation_charges ELSE 0
        END) AS q2_sum_operation_charges,

    SUM(CASE
            WHEN f.secondary_diagnosis_days > 0 OR f.secondary_diagnosis_cost > 0
            THEN f.number_of_operations ELSE 0
        END) AS q2_sum_number_of_operations,

    SUM(CASE
            WHEN f.secondary_diagnosis_days > 0 OR f.secondary_diagnosis_cost > 0
            THEN f.length_of_stay ELSE 0
        END) AS q2_sum_length_of_stay,

    SUM(CASE
            WHEN f.secondary_diagnosis_days > 0 OR f.secondary_diagnosis_cost > 0
            THEN f.admission_cost ELSE 0
        END) AS q2_sum_admission_cost,

    SUM(f.secondary_diagnosis_days) AS q2_sum_secondary_diagnosis_days,
    SUM(f.secondary_diagnosis_cost) AS q2_sum_secondary_diagnosis_cost

FROM fact_admission f
JOIN dim_time t
    ON f.time_key = t.time_key
JOIN dim_ward w
    ON f.ward_key = w.ward_key
GROUP BY
    w.ward_key,
    w.ward_name,
    t.year_num,
    t.quarter_num;

CREATE INDEX idx_mv_ward_year_quarter
    ON mv_ward_year_quarter (ward_key, year_num, quarter_num);
""")

conn.commit()

# ------------------------------------------------------------
# 4) PRINT Q1 RESULT
# Q1: For each Ward, Year, Quarter report charges per operation,
#     length of stay, and admission cost
# ------------------------------------------------------------
q1_df = pd.read_sql_query("""
SELECT
    ward_name AS ward,
    year_num AS year,
    quarter_num AS quarter,
    ROUND(
        CASE
            WHEN q1_sum_number_of_operations = 0 THEN NULL
            ELSE q1_sum_operation_charges * 1.0 / q1_sum_number_of_operations
        END, 2
    ) AS charges_per_operation,
    q1_sum_length_of_stay AS length_of_stay,
    ROUND(q1_sum_admission_cost, 2) AS admission_cost
FROM mv_ward_year_quarter
ORDER BY year_num, quarter_num, ward_name;
""", conn)

print("Q1 Result")
display(q1_df)

# ------------------------------------------------------------
# 5) PRINT Q2 RESULT
# Q2: For each Ward, Year, Quarter report charges per operation,
#     length of stay, and admission cost for Secondary Diagnosis
# ------------------------------------------------------------
q2_df = pd.read_sql_query("""
SELECT
    ward_name AS ward,
    year_num AS year,
    quarter_num AS quarter,
    ROUND(
        CASE
            WHEN q2_sum_number_of_operations = 0 THEN NULL
            ELSE q2_sum_operation_charges * 1.0 / q2_sum_number_of_operations
        END, 2
    ) AS charges_per_operation,
    q2_sum_length_of_stay AS length_of_stay,
    ROUND(q2_sum_admission_cost, 2) AS admission_cost,
    q2_sum_secondary_diagnosis_days AS secondary_diagnosis_days,
    ROUND(q2_sum_secondary_diagnosis_cost, 2) AS secondary_diagnosis_cost
FROM mv_ward_year_quarter
WHERE q2_sum_secondary_diagnosis_days > 0
   OR q2_sum_secondary_diagnosis_cost > 0
ORDER BY year_num, quarter_num, ward_name;
""", conn)

print("\nQ2 Result")
display(q2_df)

conn.close()